# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 0
%aimport _campaign_lib, api

In [2]:
import json
from _campaign_lib import *

svc = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

2026-03-10 18:18:19 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"
2026-03-10 18:18:19 INFO     [api.services.pipeline_discovery] Matched known pipeline 'termnorm'; using enriched schema
2026-03-10 18:18:19 INFO     [api.services.campaign.campaign_init] Pipeline schema loaded: termnorm vv1.1


Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93
Loaded task description: 3751 chars from LCA_INPUT_PATTERNS.md


In [3]:
campaign_config = {
    "queries_per_eval": 15,              # queries per eval step (service default: all)
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "exclude_steps": ["llm_ranking"],    # steps to skip (e.g. ["entity_profiling"])
    "pipeline_overrides": {},
    "optimization": {
        "patience": 2,                   # default: 3
        "max_rounds": 3,                 # default: 10
    },
    "eval_llm": {
        # --- Groq (free tier, open-source models) ---
        # "model": "meta-llama/llama-4-maverick-17b-128e-instruct",
        # "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",          # best quality
        "model": "claude-sonnet-4-6",      # good balance
        # "model": "claude-haiku-4-5-20251001",  # cheapest
        "provider_url": "https://api.anthropic.com",
        "max_tokens": 2000,              # response length budget
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                    "descriptions to standardized database terms using entity profiling "
                    "and candidate ranking.",
        "grid_budget": 35,               # default: 0 (full grid)
        "eval_queries_per_point": 6,     # default: 0 (all queries)
        "shared_queries": False,          # default: True
    },
}

In [4]:
#@title Pipeline snapshot (full config for reproducibility)
pipeline_config_full = await show_pipeline_snapshot(svc)

2026-03-10 18:18:19 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"


  PIPELINE SNAPSHOT: TermNorm v1.1
  Nodes:   ['fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking', 'direct_prompt']
  Schemas: ['entity_profile/1', 'llm_ranking_output/1']
  Prompts: ['entity_profiling/1', 'llm_ranking/1']

{
  "name": "TermNorm",
  "version": "v1.1",
  "available_models": [
    "meta-llama/llama-4-maverick-17b-128e-instruct",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "moonshotai/kimi-k2-instruct",
    "openai/gpt-oss-120b"
  ],
  "nodes": {
    "fuzzy_matching": {
      "type": "DeterministicFunction",
      "config": {
        "threshold": 70,
        "scorer": "WRatio",
        "limit": 5
      }
    },
    "web_search": {
      "type": "ExternalService",
      "config": {
        "max_sites": 7,
        "num_results": 20,
        "content_char_limit": 800,
        "url_fetch_multiplier": 2,
        "fallback_keywords_limit": 8,
        "query_prefix": "",
        "query_suffix": "",
        "brave_api_timeout": 10,
      

In [5]:
#@title Build pipeline params
pipeline_params = configure_pipeline(svc, campaign_config)

Active steps: ['entity_profiling', 'token_matching']
  Excluded: ['llm_ranking']


## 2. Data

In [6]:
#@title Load datasets
# Set EXCEL_PATH to load from BOM-example.xlsx; leave empty to use stored data
EXCEL_PATH = r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx"  # e.g. "../data/BOM-example.xlsx"
FORCE_RELOAD = False  # Set True to re-read Excel and overwrite stored datasets

train_data, svc["session_terms"] = prepare_datasets(
    svc["store"], svc["backend_id"],
    excel_path=EXCEL_PATH or None,
    force=FORCE_RELOAD,
)


  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ------------------------------------------------
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets


In [7]:
#@title Prepare evaluation context
campaign_rounds = []
baseline_results = []

baseline, eval_data, backend_status = await prepare_eval_context(
    svc, train_data,
)

RUN_BASELINE = False  # Set True to evaluate baseline before exploration
if RUN_BASELINE:
    campaign_rounds, baseline_results = await run_baseline_eval(
        baseline, eval_data, campaign_config, svc,
    )

2026-03-10 18:18:20 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/status "HTTP/1.1 200 OK"



BACKEND STATUS
  Session Active                 True
  Active Sessions                1
  Terms Loaded                   94
  Match Database Identifiers     109
  Match Database Aliases         599
  Experiments Count              4
  Mappings Count                 1126
  Pipeline Version               v1.1
  Llm Provider                   groq
  Llm Model                      meta-llama/llama-4-maverick-17b-128e-instruct
  ------------------------------------------------
  Experiments                   
    0_production_realtime        0 mappings
    1_production_historical      887 mappings
    2_bom_materials              159 mappings
    3_bom_processing             80 mappings

Evaluation data: 984 queries


In [8]:
#@title Candidate coverage (post-eval diagnostic)
cov_df = run_coverage_diagnostic(
    baseline_results,
    svc["store"], svc["backend_id"], svc["experiment_id"],
)

Loaded 40 eval queries
Eval runs: 18 completed runs, 4 in-progress
  run_id                name           model                      temp  accuracy  queries
  scan_15a5c1e9         scan                                      0.0   50.0%     6      
  scan_86f17bab         scan                                      0.0   66.7%     6      
  scan_fcb7bf9b         scan                                      0.0   50.0%     6      
  scan_01c3382c         scan                                      0.0   66.7%     6      
  scan_7d0c905a         scan                                      0.0   33.3%     6      
  scan_e9f03615         scan                                      0.0   66.7%     6      
  scan_3aff5881         scan                                      0.0   33.3%     6      
  scan_39b9fc27         scan                                      0.0   50.0%     6      
  scan_dff96710         scan                                      0.0   33.3%     6      
  scan_3dbc066d         scan     

## 3. Explore

Two exploration paths: **Smart Search** (scan advisor + sensitivity scan) or **Grid Search** (brute-force sweep). Use one or both.

### 3a. Smart Search

In [9]:
# preview_advisor_prompt()
preview_advisor_prompt(campaign_config, svc, task_description="TASK_DESCRIPTION", raw=True)

2026-03-10 18:18:20 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


You are an expert prompt optimization advisor. Recommend which axes (parameters and prompt fields) to prioritize in a sensitivity scan.

## Constraints (apply strictly)
- Do NOT recommend *_model axes — place them in axes_to_skip.
- Response must fit within 1500 tokens. Be terse.

## Pipeline: TermNorm AI terminology normalization pipeline
Steps execute sequentially — each step's output feeds the next:
[
  {
    "name": "cache_lookup",
    "node_role": "cache",
    "short_circuit": true
  },
  {
    "name": "fuzzy_matching",
    "node_role": "candidate_source",
    "short_circuit": true
  },
  {
    "name": "web_search",
    "node_role": "enricher"
  },
  {
    "name": "entity_profiling",
    "node_role": "enricher"
  },
  {
    "name": "token_matching",
    "node_role": "candidate_source"
  }
]

## Task Context
TASK_DESCRIPTION
## Tunable Parameters (per step)
[
  {
    "name": "fuzzy_matching",
    "param_keys": [
      "fuzzy_scorer",
      "fuzzy_threshold"
    ]
  },
  {
    "name

In [10]:
#@title Scan advisor
advisory, scan_variants, schema_labels = await run_scan_advisor(
    campaign_config, svc,
    task_description=TASK_DESCRIPTION if "TASK_DESCRIPTION" in dir() else "",
)

2026-03-10 18:18:27 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


SCAN ADVISOR -- pipeline-aware sensitivity setup
  Pipeline: termnorm (v1.1)
  Steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking']
  Excluded: ['llm_ranking']
  Task context: # Domain Context: Life Cycle Assessment (LCA) Terminology

This document capture...
  Calling claude-sonnet-4-6 ...



2026-03-10 18:18:55 INFO     [httpx] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


----------------------------------------------------------------------
PRIORITY AXES (ranked by importance)
----------------------------------------------------------------------
  1. [HIGH] query_prefix (pipeline_param) -- step: web_search
     Empty default; domain-scoped prefix steers search toward LCA/material content.
     Values: ['LCA material ecoinvent', 'life cycle assessment material database', 'ecoinvent GaBi material', 'chemical material LCA database']
  2. [HIGH] query_suffix (pipeline_param) -- step: web_search
     Empty default; suffix can anchor results to database naming conventions.
     Values: ['ecoinvent database entry', 'material classification CAS', 'LCA inventory dataset', 'GaBi ecoinvent nomenclature']
  3. [HIGH] profiling_prompt (pipeline_param) -- step: entity_profiling
     Core LLM instruction; controls depth of brand decoding and alias generation.
  4. [HIGH] profiling_schema (pipeline_param) -- step: entity_profiling
     Schema shapes token_matching in

In [12]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_variants = {
    'max_token_candidates': [10, 30, 50, 75],
    'profiling_schema': [
        [['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'], ['+', 'database_format_hint', 'string', False, "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'"]],
        [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_synonyms', 'array', False, 'Terms likely to appear verbatim in LCA database entry names for this entity'], ['+', 'no_match_signal', 'string', False, 'Brief reasoning on whether a database match is likely to exist or not']],
        [['~', 'classification_aliases', 'lca_classification_aliases', 'array', False, 'Expert-level aliases specifically aligned with LCA database naming conventions, including ecoinvent activity names and SimaPro process names'], ['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA']],
        [['+', 'lca_database_names', 'array', True, "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'"]], 
        [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions']],
        [['~', 'notes', 'material_category', 'string', True, "The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'"]]
    ],
    'profiling_temperature': [0.0, 0.3, 0.7],
    'profiling_max_tokens': [512, 1024, 2048],
    'raw_content_limit': [1000, 2500, 8000],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

  max_token_candidates: [10, 30, 50, 75]
  profiling_schema: (baseline + 6 mutations)
    [0] (baseline)
    [1] ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'), ('+', 'database_format_hint', 'string', False, 'Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'')
    [2] ('-', 'manufacturing_processes'), ('-', 'applications'), ('+', 'lca_synonyms', 'array', False, 'Terms likely to appear verbatim in LCA database entry names for this entity'), ('+', 'no_match_signal', 'string', False, 'Brief reasoning on whether a database match is likely to exist or not')
    [3] ('~', 'classification_aliases', 'lca_classification_aliases', 'array', False, 'Expert-level aliases specifically aligned with LCA database naming conventions, including ecoinvent activity names and SimaPro process names'), ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro ge

In [ ]:
#@title Build diagnostic set (with resume)
(plan_id, search_baseline, diagnostic,
 cached_profiles, variant_library) = await resume_or_build_diagnostic(
    campaign_config, baseline, baseline_results,
    svc, eval_data,
    scan_variants=scan_variants,
)

2026-03-10 10:34:31 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)
2026-03-10 10:34:31 INFO     [api.services.search.smart_search] Scan complete in plan ssplan_5ce4991c339d, reusing profiles


[RESUME] Plan ssplan_5ce4991c339d: 4 axis profiles available


In [ ]:
#@title Historical data audit & inventory
prompt_index, cached_profiles = audit_historical_data(
    svc["store"], svc["backend_id"],
    diagnostic, cached_profiles,
)

2026-03-10 10:34:31 INFO     [api.services.search.coverage] build_prompt_result_index: 18 runs -> 5 unique prompts, 57 total query results


  DATA INVENTORY  (5 prompts, 57 query results)
  Baselines: 2 plan baseline(s) -- 12 queries cached

  No axis variations found in stored plans.

  Pipeline parameters (from sensitivity scans):
    profiling_schema         4 values scanned  sensitivity: 0.300  [medium]
    max_token_candidates     3 values scanned  sensitivity: 0.150  [skip]
    profiling_temperature    3 values scanned  sensitivity: 0.150  [skip]
    relevance_weight_core    3 values scanned  sensitivity: 0.000  [skip]

  Identified: 2/5 prompts (12/57 queries) via stored plans
  Unmatched:  3 prompts (45 queries)


In [ ]:
#@title Coverage advisor
# Knobs: adjust these and re-run to see different strategies
min_queries = 6          # min queries per variant to count as "usable"
axis_requirements = None  # None = require all values; or e.g. {"persona": 2}

coverage = show_scan_coverage(
    search_baseline, variant_library, diagnostic,
    prompt_index,
    pipeline_params=campaign_config.get("pipeline_params"),
    min_queries=min_queries,
    axis_requirements=axis_requirements,
    pipeline_schema=svc.get("pipeline_schema"),
)

  COVERAGE ADVISOR  (min_queries=6)
  Baseline: 6/6 queries cached [ok]

  Pipeline params:
    max_token_candidates   3 variants | 6 cached (0 step-compatible)
    profiling_temperature  3 variants | 6 cached (0 step-compatible)
    profiling_schema       4 variants | 6 cached (0 step-compatible)
    relevance_weight_core  3 variants | 6 cached (0 step-compatible)

  Summary: 0 cached, 78 still needed (78 pipeline-param)
  >> 0/0 prompt field axes covered. Run scan to fill gaps on: max_token_candidates, profiling_temperature, profiling_schema, relevance_weight_core. Tip: lower min_queries or reduce axis_requirements to accept sparser coverage.

  Note: 57 results exist in the index under other baselines.
  The current search_baseline was rebuilt and has no cached data yet.


In [ ]:
#@title Sensitivity scan
scan_df, axis_profiles = await sensitivity_scan(
    search_baseline, variant_library, diagnostic, svc.get("backend_client"),
    user_focus=campaign_config.get("improvement_areas", ""),
    store=svc["store"], backend_id=svc["backend_id"],
    pipeline_params=campaign_config.get("pipeline_params"),
    session_terms=svc.get("session_terms"),
    plan_id=plan_id,
    prompt_result_index=prompt_index,
    pipeline_schema=svc.get("pipeline_schema"),
)

2026-03-10 10:34:32 INFO     [api.services.search.coverage] build_prompt_result_index: 18 runs -> 5 unique prompts, 57 total query results


Running sensitivity scan...
  User focus: profile schema quality, web search relevance

  Baseline field values:
    persona: You are a candidate evaluation expert.
    task_intent: Analyze entity profile and evaluate candidate matches based on core concept and ...
    problem_description: Evaluating candidates against a given entity profile and core concept.
    instruction: First, analyze the provided ENTITY PROFILE (JSON): {{entity_profile_json}} to id...
    thinking_style: Think step-by-step: Identify key features, evaluate candidates, and rank them ba...
    answer_format: {"reasoning": "[1-2 sentences explaining profile, entity_category, and key featu...

  Estimated configs: ~13 x 6 queries
  Evaluating baseline...
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 ⚡
        HIT   [token]  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra ⚡
        MISS 2/20  [token]  SJRG0010-ABS/mold

2026-03-10 10:34:32 INFO     [api.services.search.smart_search] Checkpoint: axis 'max_token_candidates' complete (1/4)


  [2] 25                                         4/6  66.7%  +15.0% ^ [cached]

  Axis 2/4: profiling_schema (pipeline_param, 4 values)
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 ⚡
        HIT   [token]  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra ⚡
        MISS 2/20  [token]  SJRG0010-ABS/molding                           -> Extrusion, plastic pipes {RER}| ext ⚡
        MISS 2/20  [token]  Kingfa NPG25                                   -> Glass fibre reinforced plastic | 60 ⚡
        MISS 11/20  [token]  PA 66 25% GF V0 RAL 7012/0                     -> Glass fibre reinforced plastic | 75 ⚡
        MISS           PA6/66 Ultramid C3U/molding                    ERR: Client error '400 Bad Request' for url ' ⚡
  [0] schema(11 fields)                          2/6  33.3%  -15.0% v [cached]
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced 

2026-03-10 10:34:32 INFO     [api.services.search.smart_search] Checkpoint: axis 'profiling_schema' complete (2/4)


  [3] schema(11 fields)                          3/6  50.0%  +0.0% [cached]

  Axis 3/4: profiling_temperature (pipeline_param, 3 values)
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 ⚡
        HIT   [token]  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra ⚡
        MISS 3/20  [token]  SJRG0010-ABS/molding                           -> Acrylonitrile-butadiene-styrene cop ⚡
        MISS 2/20  [token]  Kingfa NPG25                                   -> Glass fibre reinforced plastic | 60 ⚡
        MISS 11/20  [token]  PA 66 25% GF V0 RAL 7012/0                     -> Glass fibre reinforced plastic | 75 ⚡
        MISS 3/20  [token]  PA6/66 Ultramid C3U/molding                    -> Polyamide (Nylon) 6.6/EU-27 ⚡
  [0] 0.0                                        2/6  33.3%  -15.0% v [cached]
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic 

2026-03-10 10:34:32 INFO     [api.services.search.smart_search] Checkpoint: axis 'profiling_temperature' complete (3/4)


  [2] 0.7                                        3/6  50.0%  +0.0% [cached]

  Axis 4/4: relevance_weight_core (pipeline_param, 3 values)
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 ⚡
        HIT   [token]  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra ⚡
        MISS 3/20  [token]  SJRG0010-ABS/molding                           -> Acrylonitrile-butadiene-styrene cop ⚡
        HIT   [token]  Kingfa NPG25                                   -> Glass fibre reinforced plastic | 75 ⚡
        MISS 11/20  [token]  PA 66 25% GF V0 RAL 7012/0                     -> Glass fibre reinforced plastic | 75 ⚡
        MISS 3/20  [token]  PA6/66 Ultramid C3U/molding                    -> Polyamide (Nylon) 6.6/EU-27 ⚡
  [0] 0.5                                        3/6  50.0%  +0.0% [cached]
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 ⚡
 

2026-03-10 10:34:32 INFO     [api.services.search.smart_search] Checkpoint: axis 'relevance_weight_core' complete (4/4)
2026-03-10 10:34:32 INFO     [api.services.search.smart_search] Saved scan results to plan: ssplan_5ce4991c339d


  [2] 0.9                                        3/6  50.0%  +0.0% [cached]
  >> profiling_schema: range=30.0%, best=+15.0%, worst=-15.0%, budget=medium
  >> max_token_candidates: range=15.0%, best=+15.0%, worst=+0.0%, budget=skip
  >> profiling_temperature: range=15.0%, best=+0.0%, worst=-15.0%, budget=skip
  >> relevance_weight_core: range=0.0%, best=+0.0%, worst=+0.0%, budget=skip

Sensitivity scan complete: 13 variants evaluated

Rank  Axis                      Type            Card  Range    Budget  
----------------------------------------------------------------------
  1   profiling_schema          pipeline_param  4     0.300    medium  
  2   max_token_candidates      pipeline_param  3     0.150    skip    
  3   profiling_temperature     pipeline_param  3     0.150    skip    
  4   relevance_weight_core     pipeline_param  3     0.000    skip    


In [ ]:
#@title Select scan winner & seed campaign
best_ps, best_params = seed_campaign_from_scan(
    scan_df, axis_profiles, search_baseline, variant_library,
    campaign_rounds, campaign_config,
    store=svc["store"], backend_id=svc["backend_id"], plan_id=plan_id,
)

2026-03-10 10:34:32 INFO     [api.services.search.smart_search] select_scan_winner: 0 prompt changes, 1 param changes from 1 improving axes


Selected best from 1 improving axes:
  profiling_schema          best_delta=+15.0%  value_idx=1  acc=66.7%
Pipeline params updated: {'steps': ['entity_profiling', 'token_matching'], 'profiling_schema': {'type': 'object', 'properties': {'entity_name': {'type': 'string'}, 'core_concept': {'type': 'string', 'description': 'The single word that defines what this expression represents'}, 'distinguishing_features': {'type': 'array', 'items': {'type': 'string'}}, 'key_properties': {'type': 'array', 'items': {'type': 'string'}}, 'technical_specifications': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Explicit technical specs, dimensions, codes, ratings, tolerances'}, 'alternative_names': {'type': 'array', 'items': {'type': 'string'}}, 'classification_aliases': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Full spectrum of valid ways this entity could be referenced using expert-level terminology, from precise to generic'}, 'constituent_materials': {'type': 'ar

### 3b. Grid Search

<details>
<summary>Skip if you used Smart Search above.</summary>

Systematic sweep of the prompt configuration space. Maps the accuracy landscape before hill-climbing.

</details>

In [17]:
#@title Grid campaign overview (existing plans)
merge_plans = False  # Set True to combine results from multiple plans
grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
merged_grid_df = grid_overview.get("merged_grid_df")

Grid plans (1):
  [run..] gridplan_d603214cea60  34 points  (space=96, axes=persona,task_intent,thinking_style,answer_format,problem_description)
Loaded 40 eval queries
Eval runs: 18 completed runs, 4 in-progress
  run_id                name           model                      temp  accuracy  queries
  scan_15a5c1e9         scan                                      0.0   50.0%     6      
  scan_86f17bab         scan                                      0.0   66.7%     6      
  scan_fcb7bf9b         scan                                      0.0   50.0%     6      
  scan_01c3382c         scan                                      0.0   66.7%     6      
  scan_7d0c905a         scan                                      0.0   33.3%     6      
  scan_e9f03615         scan                                      0.0   66.7%     6      
  scan_3aff5881         scan                                      0.0   33.3%     6      
  scan_39b9fc27         scan                                      0

In [18]:
#@title Build or resume grid plan
gs = campaign_config["grid_search"]

llm_client, llm_model = setup_llm(campaign_config)

(
    grid_plan_id, grid_points, grid_state_lookup,
    grid_axes, layer1_fields, grid_baseline,
) = await resume_or_build_grid(
    campaign_config, baseline, llm_client, llm_model,
    svc["store"], svc["backend_id"],
    improvement_areas=campaign_config.get("improvement_areas", ""),
)

print(f"Grid points: {len(grid_points)}")
print(f"Plan ID: {grid_plan_id}")

2026-03-10 10:34:59 INFO     [api.services.search.grid_core] Resuming grid plan gridplan_d603214cea60 (status: in_progress, 34 points)


[RESUME] Found existing grid plan: gridplan_d603214cea60
  Grid points: 34
Grid points: 34
Plan ID: gridplan_d603214cea60


In [19]:
#@title Run grid search
grid_df = await run_grid_search(
    grid_points, grid_state_lookup, eval_data,
    campaign_config["eval_llm"],
    plan_id=grid_plan_id,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_client=svc.get("backend_client"),
    session_terms=svc.get("session_terms"),
    pipeline_params=campaign_config.get("pipeline_params"),
    eval_queries_per_point=gs.get("eval_queries_per_point", 1),
    shared_queries=gs.get("shared_queries", False),
    grid_seed=gs.get("seed", 42),
)

2026-03-10 10:35:02 INFO     [httpx] HTTP Request: POST http://127.0.0.1:8000/sessions "HTTP/1.1 200 OK"


[resume] Skipping 1/34 stored grid points, evaluating 33 remaining
  [1/34] acc=50.0% (3/6)


2026-03-10 10:35:03 INFO     [httpx] HTTP Request: POST http://127.0.0.1:8000/matches "HTTP/1.1 200 OK"


        MISS 5/20  [token]  EN 10270-3-1.4310-NS-1/cold forming            -> Wire drawing, steel {RER}| wire dra 1.5s


2026-03-10 10:35:06 INFO     [httpx] HTTP Request: POST http://127.0.0.1:8000/matches "HTTP/1.1 200 OK"


...

        MISS           PE                                             ERR: Client error '404 Not Found' for url 'ht


2026-03-10 10:35:09 INFO     [httpx] HTTP Request: POST http://127.0.0.1:8000/matches "HTTP/1.1 404 Not Found"
2026-03-10 10:35:09 WARNING  [api.services.prompt_eval] backend_reranker_eval failed for EN 10088-2 - X5CrNi18-10+2R (1.4301)
BASF Ultrason E2010 G/s: Client error '404 Not Found' for url 'http://127.0.0.1:8000/matches'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404


        MISS           EN 10088-2 - X5CrNi18-10+2R (1.4301)
BASF Ult  ERR: Client error '404 Not Found' for url 'ht


2026-03-10 10:35:10 WARNING  [api.services.search.grid_core] Grid search interrupted at point 1/34. Completed points are saved via evaluate_prompt_cached.



  [INTERRUPTED] Grid search
  Completed: 1/34 grid points
  Saved: completed points saved via evaluate_prompt_cached (content-hash dedup)
  Resume: re-run this cell -- stored points will be skipped automatically


In [ ]:
#@title Display grid results
_display_df = merged_grid_df if merged_grid_df is not None else grid_df
display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [ ]:
#@title LLM analysis of grid results
_analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
llm_client, llm_model = setup_llm(campaign_config)
grid_analysis = await analyze_grid_results(
    _analysis_df, grid_axes, llm_client, model=llm_model,
)

In [ ]:
#@title Select grid winner and seed campaign
grid_winner = select_and_seed_grid_winner(
    grid_df, merged_grid_df, grid_state_lookup,
    grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
)

## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [ ]:
#@title Run optimization (feedback cycle — M3 nodes)
campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_url=svc["backend_client"].base_url,
    pipeline_params=campaign_config.get("pipeline_params"),
    session_terms=svc.get("session_terms"),
)

In [ ]:
#@title Run optimization round (manual)
round_entry = await run_manual_round(
    campaign_rounds, eval_data, campaign_config, svc,
)

## 5. Results

In [ ]:
#@title Campaign comparison table
show_campaign_summary(campaign_rounds)

In [ ]:
#@title Per-query flip tracking (baseline vs final)
show_flip_tracking(campaign_rounds)

In [ ]:
#@title PromptState lineage chain
show_lineage_chain(campaign_rounds)

In [ ]:
#@title Save winner
save_campaign_winner(campaign_rounds, campaign_config, svc["store"], svc["backend_id"])

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print("--- SUGGESTED CONFIG (copy to Setup) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

In [ ]:
#@title Sync evaluation history to Langfuse
# Safe to re-run — already-pushed runs are skipped automatically.
stats = sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)